In [ ]:
%reload_ext autoreload # Automatically reload modules before executing code in Jupyter cells
%autoreload 2 # Automatically reload modules before executing code in Jupyter cells
%reload_ext jupyter_black # Auto-format code in Jupyter cells using the Black code formatter

# Drugs

In [ ]:
year = "<UMLS-year>"
version = "<UMLS-version>"
umls_zip_path = f"<your_path>/umls-{year}{version}.zip"

In [ ]:
import zipfile

import pandas as pd
from tqdm import tqdm

atc_cuis = {"CUI": [], "ATC": []}
path = umls_zip_path
with zipfile.ZipFile(path) as zip_file:
    with zip_file.open(f"{year}{version}/META/MRCONSO.RRF", mode="r") as file:
        lines = file.readlines()
        print(f"ALL UMLS {year}{version}: {len(lines)} concepts")
        for line in tqdm(lines):
            line = str(line)[2:-3].split("|")
            if line[11] == "ATC":
                atc_cuis["CUI"].append(line[0])
                atc_cuis["ATC"].append(line[13])
atc_cuis_df = pd.DataFrame(atc_cuis).drop_duplicates()
print(f"ATC {year}{version}: {len(atc_cuis_df)} cui/code combinations")

In [ ]:
import zipfile

import pandas as pd
from tqdm import tqdm

med_cuis = set(atc_cuis_df["CUI"].unique())
med_syn = dict(CUI=[], STR=[])
path = umls_zip_path
with zipfile.ZipFile(path) as zip_file:
    with zip_file.open(f"{year}{version}/META/MRCONSO.RRF", mode="r") as file:
        lines = file.readlines()
        print(f"ALL UMLS {year}{version}: {len(lines)} concepts")
        for line in tqdm(lines):
            line = str(line)[2:-3].split("|")
            if line[1] in ["FRE", "ENG"]:
                if line[0] in med_cuis:
                    med_syn["CUI"].append(line[0])
                    med_syn["STR"].append(line[14])
med_syn_df = pd.DataFrame(med_syn).drop_duplicates()
print(f"ATC {year}{version}: {len(med_syn_df)} synonymes")

In [ ]:
umls_atc = med_syn_df.merge(atc_cuis_df, on="CUI")[["ATC", "STR"]]
umls_atc.to_csv(f"atc_str_{year}{version}.csv", index=False)

In [ ]:
import pandas as pd

umls_atc = pd.read_csv(f"atc_str_{year}{version}.csv").explode("STR")[["ATC", "STR"]]
romedi_atc = pd.DataFrame.from_dict(
    dict(
        pd.read_pickle(
            "../drug_knowledge/final_dict.pkl"
        )
    ),
    orient="index",
)
romedi_atc = (
    pd.DataFrame(romedi_atc.stack().groupby(level=0).agg(set), columns=["STR"])
    .explode("STR")
    .reset_index()
    .rename(columns={"index": "ATC"})
)
full_atc_str = pd.concat([romedi_atc, umls_atc]).drop_duplicates()
full_atc_str.to_csv(f"full_atc_str_{year}{version}.csv", index=False)

# Laboratory tests

In [ ]:
import zipfile

from tqdm import tqdm

snomed_cuis = []
path = umls_zip_path
with zipfile.ZipFile(path) as zip_file:
    with zip_file.open(f"{year}{version}/META/MRCONSO.RRF", mode="r") as file:
        lines = file.readlines()
        print(f"ALL UMLS {year}{version}: {len(lines)} concepts")
        for line in tqdm(lines):
            line = str(line)[2:-3].split("|")
            if line[11] == "SNOMEDCT_US":
                snomed_cuis.append(line[0])
snomed_cuis = set(snomed_cuis)
print(f"SNOMED CT US {year}{version}: {len(snomed_cuis)} concepts")

In [ ]:
import zipfile

import pandas as pd
from tqdm import tqdm

bio_cuis = []
path = umls_zip_path
with zipfile.ZipFile(path) as zip_file:
    with zip_file.open(f"{year}{version}/META/MRSTY.RRF", mode="r") as file:
        lines = file.readlines()
        for line in tqdm(lines):
            line = str(line)[2:-3].split("|")
            if line[1] == "T059":
                bio_cuis.append(line[0])
bio_cuis = set(bio_cuis)
print(f"LABORATORY PROCEDURE {year}{version}: {len(bio_cuis)} concepts")

In [ ]:
bio_snomed_cuis = snomed_cuis.intersection(bio_cuis)
print(f"LAB SNOMED {year}{version}: {len(bio_snomed_cuis)} concepts")

In [ ]:
import zipfile

import pandas as pd
from tqdm import tqdm

snomed_syn = dict(CUI=[], STR=[])
path = umls_zip_path
with zipfile.ZipFile(path) as zip_file:
    with zip_file.open(f"{year}{version}/META/MRCONSO.RRF", mode="r") as file:
        lines = file.readlines()
        print(f"ALL UMLS {year}{version}: {len(lines)} concepts")
        for line in tqdm(lines):
            line = str(line)[2:-3].split("|")
            if line[1] in ["FRE", "ENG"]:
                if line[0] in bio_snomed_cuis:
                    snomed_syn["CUI"].append(line[0])
                    snomed_syn["STR"].append(line[14])
snomed_syn_df = pd.DataFrame(snomed_syn).drop_duplicates()
print(f"SNOMED CT US {year}{version}: {len(snomed_syn_df)} synonymes")

In [ ]:
snomed_syn_df.to_csv(f"lab_snomed_ct_{year}{version}.csv", index=False)